In [1]:
# %% [markdown]
# # Word-Level SPLADE: Tokenizer Integration
# 
# This notebook demonstrates the integration between our word-level vocabulary 
# and SPLADE model architecture. We'll test tokenization, model creation, 
# and basic encoding functionality.

# %% [markdown]
# ## 1. Setup and Imports

# %%
import sys
from pathlib import Path
import torch
import numpy as np

# Add project root
project_root = Path('/work')
sys.path.append(str(project_root))

# Import our components
from sandbox.splade.core.vocabulary import VocabularyBuilder, WordTokenizer
from sandbox.splade.core.models import (
    WordLevelTokenizer, 
    WordSpladeModel, 
    WordSpladeEncoder,
    create_dual_word_splade_system,
    create_word_splade_system,
    test_tokenization,
    test_model_forward
)

print("📦 Imports successful!")
print(f"🖥️  Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# %% [markdown]
# ## 2. Load Vocabulary
# 
# First, let's load the vocabulary we built in Step 1.

# %%
# Configuration
VOCAB_BASE_PATH = Path("/work/sandbox/splade/vocabularies")
FILE_BASE = "OEB"
VOCAB_SIZE = 50000
USE_FILTERED = True  # Set to False to use full vocabulary (with stopwords)

# Determine vocabulary path based on choice
vocab_suffix = "filtered" if USE_FILTERED else "full"
VOCAB_PATH = VOCAB_BASE_PATH / f"{FILE_BASE}_vocab_{vocab_suffix}_{VOCAB_SIZE}"
print(f"📁 Using {'filtered' if USE_FILTERED else 'full'} vocabulary")
print(f"📍 Vocabulary path: {VOCAB_PATH}")

BASE_MODEL = "dccuchile/bert-base-spanish-wwm-cased"  # Spanish BERT

# Load vocabulary
print("📚 Loading vocabulary...")
try:
    vocab_builder = VocabularyBuilder.load_vocabulary(VOCAB_PATH)
    print(f"✅ Vocabulary loaded: {len(vocab_builder.word_to_id):,} words")
    print(f"📊 Special tokens: {list(vocab_builder.special_tokens.keys())}")
except Exception as e:
    print(f"❌ Error loading vocabulary: {e}")
    print("💡 Make sure you've run the vocabulary building step first!")

# %% [markdown]
# ## 2.5. Compare Full vs Filtered Vocabularies
# 
# Let's test both vocabulary approaches to see the differences.

# %%
# Test both vocabularies if available
print("🔍 Comparing vocabulary approaches:")
print("="*50)

for use_filt, name in [(False, "Full"), (True, "Filtered")]:
    vocab_suffix = "filtered" if use_filt else "full"
    vocab_path_test = VOCAB_BASE_PATH / f"{FILE_BASE}_vocab_{vocab_suffix}_{VOCAB_SIZE}"
    
    if vocab_path_test.exists():
        print(f"\n📚 Loading {name} vocabulary...")
        try:
            vocab_test = VocabularyBuilder.load_vocabulary(vocab_path_test)
            print(f"  ✅ {name} vocabulary: {len(vocab_test.word_to_id):,} words")
            
            # Test stopword inclusion
            spanish_stopwords = ['de', 'la', 'el', 'en', 'con', 'para', 'por']
            found_stopwords = [w for w in spanish_stopwords if w in vocab_test.word_to_id]
            print(f"  🔤 Sample stopwords found: {found_stopwords}")
            
        except Exception as e:
            print(f"  ❌ Error loading {name} vocabulary: {e}")
    else:
        print(f"\n⚠️  {name} vocabulary not found at {vocab_path_test}")

print(f"\n💡 Using: {'Filtered' if USE_FILTERED else 'Full'} vocabulary for this session")

# %% [markdown]
# ## 3. Create Word-Level Tokenizer
# 
# Now let's create and test our word-level tokenizer.

# %%
# Create tokenizer
print("🔤 Creating word-level tokenizer...")
tokenizer = WordLevelTokenizer(vocab_builder, max_length=512)

print(f"✅ Tokenizer created!")
print(f"📊 Vocabulary size: {tokenizer.vocab_size:,}")
print(f"📏 Max length: {tokenizer.max_length}")

# %% [markdown]
# ## 4. Test Tokenization
# 
# Let's test the tokenizer with some technical Spanish text to see how it handles domain-specific terms.

# %%
# Test with technical medical terms
test_texts = [
    "Canalización hormigonada de  1  tubos de polietileno libre de halógenos de 110 mm de diámetro en andén, incluso la demolición y reposición del pavimento y la solera, el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Nocturno Excepcional Banda de mantenimiento: No necesita intervalo Condiciones de ejecución: Volumen relevante",
    "Canalización hormigonada de  5  tubos de polietileno libre de halógenos de 110 mm de diámetro en andén, incluso la demolición y reposición del pavimento y la solera, el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Nocturno Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Volumen relevante",
    "Canalización hormigonada de  16  tubos de polietileno libre de halógenos de 160 mm de diámetro adosada o superficial, incluso  el relleno y compactado de la zanja, la preparación y la nivelación de la solera, el suministro y montaje del encofrado, elementos de fijación, desencofrado, el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Cualquier franja horaria Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Volumen escaso",
    "Suministro de canalización de 1 o 2 tubos de acero galvanizado de 100mm de diámetro para instalaciones CMS en transiciones entre diferentes plataformas.  Trabajo: Diurno Excepcional Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Cualquier condición de ejecución",
    "Canalización hormigonada de  5  tubos de polietileno libre de halógenos de 50 mm de diámetro en cualquier clase de terreno, excepto roca, incluso  el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Diurno Excepcional Banda de mantenimiento: No necesita intervalo Condiciones de ejecución: Volumen relevante",
    "Canalización hormigonada de  8  tubos de polietileno libre de halógenos de 90 mm de diámetro en cruce bajo vías, incluso el descerne y la entibación de los costados y la posterior reposición del balasto retirado, el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Nocturno Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Volumen relevante"
]

test_tokenization(tokenizer, test_texts)

# %% [markdown]
# ## 5. Test Encoding/Decoding Roundtrip
# 
# Let's verify that our tokenizer can encode and decode text properly.

# %%
# Test encoding/decoding roundtrip
test_text = "Suministro de canalización de 1 o 2 tubos de acero galvanizado de 100mm de diámetro para instalaciones CMS en transiciones entre diferentes plataformas.  Trabajo: Diurno Excepcional Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Cualquier condición de ejecución"

print("🔄 Testing encoding/decoding roundtrip:")
print(f"📝 Original: {test_text}")

# Encode
encoding = tokenizer(test_text, padding=True, truncation=True, return_tensors="pt")
print(f"🔢 Encoded shape: {encoding['input_ids'].shape}")
print(f"🔢 Token IDs tensor: {encoding['input_ids']}")
print(f"🔢 Token IDs list: {encoding['input_ids'][0].tolist()}")

# Debug: Check what type we're getting
token_ids = encoding['input_ids'][0].tolist()
print(f"🐛 Debug - type of token_ids: {type(token_ids)}")
print(f"🐛 Debug - token_ids value: {token_ids}")
print(f"🐛 Debug - is it a list?: {isinstance(token_ids, list)}")

# Decode back
decoded = tokenizer.decode(encoding['input_ids'][0], skip_special_tokens=True)
print(f"🔄 Decoded: {decoded}")

# Check if important terms are preserved
important_terms = ["suministro", "canalización", "tubos", "acero", "galvanizado"]
preserved = [term for term in important_terms if term.lower() in decoded.lower()]
print(f"🎯 Preserved terms: {preserved}")

# %% [markdown]
# ## 6. Create SPLADE Model
# 
# Now let's create the word-level SPLADE model architecture.

# %%
# Create Word SPLADE model
print("🧠 Creating Word-Level SPLADE model...")

try:
    model = WordSpladeModel(
        base_model_name=BASE_MODEL,
        vocab_size=tokenizer.vocab_size,
        hidden_size=768,  # Will be auto-detected
        dropout=0.1
    )
    
    print(f"✅ Model created successfully!")
    print(f"📊 Model vocabulary size: {model.vocab_size:,}")
    print(f"🧠 Hidden size: {model.hidden_size}")
    print(f"⚖️  Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print(f"🖥️  Model moved to: {device}")
    
except Exception as e:
    print(f"❌ Error creating model: {e}")
    print("💡 Check if the base model is available and vocabulary is loaded correctly")

# %% [markdown]
# ## 7. Test Model Forward Pass
# 
# Let's test if our model can process text and produce sparse representations.

# %%
# Test model forward pass
test_model_forward(model, tokenizer, 
                  test_text="Canliación de dos conductos en cualquier tipo de terreno")

# %% [markdown]
# ## 8. Create Complete SPLADE System
# 
# Let's use the convenience function to create a complete system.

# %%
# Create complete system (alternative approach)
print("⚡ Creating complete Word-Level SPLADE system...")

try:
    # This creates model, tokenizer, and encoder in one go
    model_alt, tokenizer_alt, encoder = create_dual_word_splade_system(
        vocab_base_path=VOCAB_BASE_PATH,
        file_base=FILE_BASE,
        base_model_name=BASE_MODEL,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        use_filtered=USE_FILTERED
    )
    
    print("✅ Complete system created!")
    
except Exception as e:
    print(f"❌ Error creating system: {e}")
    # Fallback to direct creation
    model_alt, tokenizer_alt, encoder = create_word_splade_system(
        vocab_path=VOCAB_PATH,
        base_model_name=BASE_MODEL,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

# %% [markdown]
# ## 9. Test Encoding with Different Text Lengths
# 
# Let's test how our system handles texts of different lengths.

# %%
# Test with different text lengths
test_cases = [
    # Short text
    "Zanja a mano en balasto",
    
    # Medium text  
    "Canalización de un conducto bajo vías.",
    
    # Long text
    """Canalización hormigonada de  1  tubos de polietileno libre de halógenos de 90 mm de diámetro en cruce bajo vías, incluso el descerne y la entibación de los costados y la posterior reposición del balasto retirado, el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Nocturno Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Volumen escaso"""
]

print("📏 Testing encoding with different text lengths:")
print("="*60)

for i, text in enumerate(test_cases, 1):
    print(f"\n🧪 Test case {i}:")
    print(f"📝 Text ({len(text)} chars): {text[:100]}{'...' if len(text) > 100 else ''}")
    
    # Encode with our system
    with torch.no_grad():
        sparse_repr = encoder.encode([text])
        
    print(f"🔢 Output shape: {sparse_repr.shape}")
    print(f"⚡ Non-zero elements: {torch.sum(sparse_repr > 0).item()}")
    print(f"📊 Sparsity: {1 - (torch.sum(sparse_repr > 0).item() / sparse_repr.numel()):.4f}")
    print(f"📈 Max activation: {torch.max(sparse_repr).item():.4f}")

# %% [markdown]
# ## 10. Compare Word vs Sub-word Tokenization
# 
# Let's visualize the difference between our word-level approach and sub-word tokenization.

# %%
# Compare tokenization approaches
from transformers import AutoTokenizer

# Load a sub-word tokenizer for comparison
subword_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

comparison_texts = [
    "canalización",
    "hormigonada", 
    "acero",
    "galvanizado",
    "bajo vías"
]

print("🔍 Word-level vs Sub-word Tokenization Comparison:")
print("="*70)

for text in comparison_texts:
    print(f"\n📝 Text: '{text}'")
    
    # Word-level tokenization
    word_tokens = tokenizer.word_tokenizer.tokenize(text)
    print(f"📖 Word-level:  {word_tokens} ({len(word_tokens)} tokens)")
    
    # Sub-word tokenization
    subword_tokens = subword_tokenizer.tokenize(text)
    print(f"🔤 Sub-word:    {subword_tokens} ({len(subword_tokens)} tokens)")
    
    # Analysis
    if len(word_tokens) < len(subword_tokens):
        print(f"✅ Word-level preserves semantic unity ({len(word_tokens)} vs {len(subword_tokens)})")
    else:
        print(f"⚠️  Sub-word more compact ({len(subword_tokens)} vs {len(word_tokens)})")

# %% [markdown]
# ## 11. Vocabulary Coverage Analysis
# 
# Let's analyze how well our vocabulary covers technical terms vs general terms.

# %%
# Analyze vocabulary coverage on different types of text
technical_texts = [
    "canalización hormigonada tubo PVC polietileno acero",
    "zanja hinca cata relleno transición conexión entronque",
    "tubería drenaje arqueta cámara canaleta túnel",
    "tubo corrugado alma acero galvanizado presión",
    "mandrilado limpieza localización reparación metálica"
]

general_texts = [
    "La correcta nivelación del terreno es fundamental para garantizar el flujo gravitacional en canalizaciones de saneamiento",
    "Antes de la instalación, es imprescindible realizar un estudio geotécnico para definir el tipo de zanja y el sistema de entibación adecuado",
    "Las canalizaciones deben diseñarse teniendo en cuenta el tipo de fluido, el caudal previsto y la presión de trabajo",
    "Durante la ejecución, es vital asegurar la compactación del relleno en capas para evitar asentamientos futuros",
    "La señalización y protección de las zanjas abiertas son obligatorias para garantizar la seguridad de los trabajadores y peatones"
]

def analyze_coverage(texts, text_type):
    print(f"\n📊 Coverage Analysis - {text_type}:")
    total_tokens = 0
    covered_tokens = 0
    oov_words = set()
    
    for text in texts:
        tokens = tokenizer.word_tokenizer.tokenize(text)
        for token in tokens:
            total_tokens += 1
            if token in vocab_builder.word_to_id:
                covered_tokens += 1
            else:
                oov_words.add(token)
    
    coverage = covered_tokens / total_tokens if total_tokens > 0 else 0
    print(f"  Coverage: {coverage:.3f} ({covered_tokens}/{total_tokens})")
    print(f"  OOV words: {len(oov_words)} unique")
    if oov_words:
        print(f"  Examples: {list(oov_words)[:10]}")

analyze_coverage(technical_texts, "Technical Construction Text")
analyze_coverage(general_texts, "General Construction Text")

# %% [markdown]
# ## 12. Memory and Performance Analysis
# 
# Let's check memory usage and encoding performance.

# %%
import time
import psutil
import os

def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

# Performance test
test_texts_perf = [
    "Canalización de 1 conducto bajo vías." 
] * 100  # 100 identical texts for testing

print("⚡ Performance Analysis:")
print("="*40)

# Memory before
mem_before = get_memory_usage()
print(f"💾 Memory before: {mem_before:.1f} MB")

# Encoding performance
start_time = time.time()
with torch.no_grad():
    encoded_batch = encoder.encode(test_texts_perf, batch_size=16)
end_time = time.time()

# Memory after
mem_after = get_memory_usage()
print(f"💾 Memory after: {mem_after:.1f} MB")
print(f"📈 Memory increase: {mem_after - mem_before:.1f} MB")

# Performance metrics
total_time = end_time - start_time
texts_per_sec = len(test_texts_perf) / total_time
print(f"⏱️  Encoding time: {total_time:.2f} seconds")
print(f"🚀 Throughput: {texts_per_sec:.1f} texts/second")
print(f"📊 Output shape: {encoded_batch.shape}")
print(f"⚡ Average sparsity: {1 - (torch.sum(encoded_batch > 0).item() / encoded_batch.numel()):.4f}")

# %% [markdown]
# ## 13. Sparse Representation Analysis
# 
# Let's analyze the sparse representations in detail.

# %%
# Analyze sparse representations
analysis_text = "Canalización hormigonada de 2 conductos en cualquier tipo de terreno"

print("🔬 Sparse Representation Analysis:")
print("="*50)
print(f"📝 Text: {analysis_text}")

# Get sparse representation
with torch.no_grad():
    sparse_vec = encoder.encode([analysis_text])[0]  # Single vector

# Basic statistics
non_zero_mask = sparse_vec > 0
num_nonzero = torch.sum(non_zero_mask).item()
total_vocab = len(sparse_vec)
sparsity = 1 - (num_nonzero / total_vocab)

print(f"\n📊 Statistics:")
print(f"  Vocabulary size: {total_vocab:,}")
print(f"  Non-zero elements: {num_nonzero:,}")
print(f"  Sparsity: {sparsity:.4f} ({sparsity*100:.2f}%)")
print(f"  Max value: {torch.max(sparse_vec).item():.4f}")
print(f"  Mean of non-zero: {torch.mean(sparse_vec[non_zero_mask]).item():.4f}")

# Top activated words
top_k = 20
top_values, top_indices = torch.topk(sparse_vec, k=top_k)

print(f"\n🔥 Top {top_k} activated words:")
for i, (idx, val) in enumerate(zip(top_indices, top_values)):
    word = vocab_builder.id_to_word.get(idx.item(), "[UNK]")
    print(f"  {i+1:2d}. {word:20s} -> {val.item():.4f}")

# %% [markdown]
# ## 14. Save Model State (Optional)
# 
# Save the current model state for later use.

# %%
# Save model and configuration
SAVE_PATH = Path("/work/splade_word/models/word_splade_v1")

print("💾 Saving model state...")
try:
    # Save model
    model.save_pretrained(SAVE_PATH)
    
    # Save additional metadata
    metadata = {
        "vocab_path": str(VOCAB_PATH),
        "base_model": BASE_MODEL,
        "vocab_size": tokenizer.vocab_size,
        "max_length": tokenizer.max_length,
        "created_date": str(torch.datetime.now() if hasattr(torch, 'datetime') else 'unknown'),
        "device": str(device)
    }
    
    import json
    with open(SAVE_PATH / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✅ Model saved to: {SAVE_PATH}")
    
except Exception as e:
    print(f"❌ Error saving model: {e}")

# %% [markdown]
# ## 15. Summary and Next Steps
# 
# Let's summarize what we've accomplished and outline next steps.

# %%
print("🎉 TOKENIZER INTEGRATION COMPLETE!")
print("="*50)

print("\n✅ What we accomplished:")
print("  1. ✅ Created word-level tokenizer integrated with custom vocabulary")
print("  2. ✅ Built SPLADE model architecture for word-level tokens")  
print("  3. ✅ Tested tokenization preserves technical terms")
print("  4. ✅ Verified model can produce sparse representations")
print("  5. ✅ Analyzed performance and memory usage")
print("  6. ✅ Compared word-level vs sub-word tokenization")

print(f"\n📊 System specifications:")
print(f"  • Vocabulary size: {tokenizer.vocab_size:,} words")
print(f"  • Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  • Device: {device}")
print(f"  • Max sequence length: {tokenizer.max_length}")

print(f"\n🎯 Key benefits observed:")
print(f"  • Technical terms preserved as single tokens")
print(f"  • Sparse representations maintain semantic meaning")
print(f"  • Good coverage on technical vocabulary")
print(f"  • Reasonable performance and memory usage")

print(f"\n🚀 Next steps:")
print(f"  1. Train the model on your document collection")
print(f"  2. Build search indices with word-level SPLADE")
print(f"  3. Compare retrieval performance vs sub-word SPLADE")
print(f"  4. Fine-tune sparsity and performance parameters")

print(f"\n💡 Ready to proceed to training phase!")

# %% [markdown]
# ## 16. Quick Testing Function for Future Use
# 
# A utility function to quickly test the system with new texts.

# %%
def quick_test_word_splade(text: str, show_top_words: int = 10, compare_vocabularies: bool = False):
    """
    Quick utility to test word SPLADE encoding on any text.
    Perfect for interactive testing in notebooks.
    """
    print(f"🧪 Quick Word-SPLADE Test")
    print(f"📝 Input: {text}")
    
    # Tokenize
    tokens = tokenizer.word_tokenizer.tokenize(text)
    print(f"🔤 Tokens: {tokens} ({len(tokens)} words)")
    
    # Encode
    with torch.no_grad():
        sparse_vec = encoder.encode([text])[0]
    
    # Statistics
    non_zero = torch.sum(sparse_vec > 0).item()
    sparsity = 1 - (non_zero / len(sparse_vec))
    
    print(f"📊 Non-zero: {non_zero:,} | Sparsity: {sparsity:.3f} | Max: {torch.max(sparse_vec).item():.3f}")
    
    # Top words
    if show_top_words > 0:
        top_vals, top_idxs = torch.topk(sparse_vec, k=show_top_words)
        print(f"🔥 Top words:")
        for i, (idx, val) in enumerate(zip(top_idxs, top_vals)):
            word = vocab_builder.id_to_word.get(idx.item(), "[UNK]")
            print(f"  {word}: {val.item():.3f}")

    # Optional vocabulary comparison
    if compare_vocabularies:
        print(f"\n🔄 Vocabulary Comparison:")
        stopwords_in_tokens = [t for t in tokens if t.lower() in ['de', 'la', 'el', 'en', 'con', 'para', 'por']]
        if stopwords_in_tokens:
            print(f"  Stopwords in text: {stopwords_in_tokens}")
            print(f"  Current vocab type: {'Filtered' if USE_FILTERED else 'Full'}")

# Test the function
print("🔧 Testing quick utility function:")
quick_test_word_splade("Construcción de zanja a mano, en cualquier tipo de terreno", show_top_words=8)

📦 Imports successful!
🖥️  Device available: CUDA
📁 Using filtered vocabulary
📍 Vocabulary path: /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
📚 Loading vocabulary...
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
Vocabulary size: 333
✅ Vocabulary loaded: 333 words
📊 Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
🔍 Comparing vocabulary approaches:

📚 Loading Full vocabulary...
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_full_50000
Vocabulary size: 361
  ✅ Full vocabulary: 361 words
  🔤 Sample stopwords found: ['de', 'la', 'el', 'en', 'con', 'para', 'por']

📚 Loading Filtered vocabulary...
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
Vocabulary size: 333
  ✅ Filtered vocabulary: 333 words
  🔤 Sample stopwords found: []

💡 Using: Filtered vocabulary for this session
🔤 Creating word-level tokenizer...
✅ Tokenizer created!
📊 Vocabulary size: 333
📏 Max length: 512
🧪 Tes

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bi

✅ Model created successfully!
📊 Model vocabulary size: 333
🧠 Hidden size: 768
⚖️  Model parameters: 110,106,957
🖥️  Model moved to: cuda
🧪 Testing Model Forward Pass
🖥️  Model device: cuda:0
📝 Input text: Canliación de dos conductos en cualquier tipo de terreno
📊 Input shape: torch.Size([1, 512])
🖥️  Input device: cuda:0
🔢 Logits shape: torch.Size([1, 512, 333])
⚡ Sparse repr shape: torch.Size([1, 333])
🎯 Non-zero elements: 235
📈 Max value: 2.1952

🔥 Top activated words:
   carga: 2.1952
   CATA: 2.1153
   at: 2.0295
   cable: 1.9666
   mandrilado: 1.9650
   cama: 1.8905
   sellado: 1.8803
   3 <= i < 5 horas: 1.8351
   nocturno: 1.7022
   tensor: 1.6822
⚡ Creating complete Word-Level SPLADE system...
📁 Loading filtered vocabulary from: /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
🔧 Creating Word-Level SPLADE System...
📚 Loading vocabulary...
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
Vocabulary size: 333
🔤 Creating tokenizer...
🧠 Cr

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bi

⚡ Creating encoder...
WordSpladeEncoder initialized on cuda
Vocabulary size: 333
✅ Word-Level SPLADE system ready!
📊 Vocabulary size: 333
🖥️  Device: cuda
✅ Complete system created!
📏 Testing encoding with different text lengths:

🧪 Test case 1:
📝 Text (23 chars): Zanja a mano en balasto
🔢 Output shape: torch.Size([1, 333])
⚡ Non-zero elements: 190
📊 Sparsity: 0.4294
📈 Max activation: 2.1313

🧪 Test case 2:
📝 Text (38 chars): Canalización de un conducto bajo vías.
🔢 Output shape: torch.Size([1, 333])
⚡ Non-zero elements: 244
📊 Sparsity: 0.2673
📈 Max activation: 1.9263

🧪 Test case 3:
📝 Text (509 chars): Canalización hormigonada de  1  tubos de polietileno libre de halógenos de 90 mm de diámetro en cruc...
🔢 Output shape: torch.Size([1, 333])
⚡ Non-zero elements: 194
📊 Sparsity: 0.4174
📈 Max activation: 4.1675
🔍 Word-level vs Sub-word Tokenization Comparison:

📝 Text: 'canalización'
📖 Word-level:  ['canalización'] (1 tokens)
🔤 Sub-word:    ['canal', '##ización'] (2 tokens)
✅ Word-level 